# 02 — RAG: teaching your agent to read *your* documents

In notebook 01 you gave the agent tools. `search_web` let it read the **internet**.
RAG (**R**etrieval-**A**ugmented **G**eneration) is the same idea pointed at **your own documents**.

The shape is identical to `search_web`:
> retrieve relevant text  →  feed it to Claude  →  let it answer.

Only two things change: the **source** (your files, not the web) and the **search method**
(by *meaning*, not keywords).

By the end you'll have a `search_docs` tool the agent calls on its own — the exact
three-piece pattern (**function → schema → registry**) you already know.

## 0. Install (first run only)

`sentence-transformers` gives us a small embedding model that runs **locally and free — no API key**
(Claude has no embeddings endpoint; embeddings come from a separate model).
The first time you call `.encode()` it downloads a ~90 MB model.

In [ ]:
%pip install -q sentence-transformers

## 1. Setup — same client as notebook 01

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

import anthropic

client = anthropic.Anthropic()
model = "claude-haiku-4-5"   # same model you used in notebook 01

## 2. A tiny knowledge base

These five short documents stand in for *your* private data — a company handbook, product
docs, your own notes. (Swap in real files later — see the last cell.)

This is a fictional online coffee company, **Brewhaus**.

In [ ]:
documents = {
    "Refund Policy": (
        "If you are unhappy with an order, you can return it within 30 days for a full "
        "refund. Opened bags are eligible as long as at least half the bag remains.\n\n"
        "Refunds go back to your original payment method within 5 business days. "
        "We do not charge a restocking fee."
    ),
    "Account Recovery": (
        "To regain access when you cannot sign in, click 'Forgot credentials' on the "
        "login page. We email a secure reset link that expires after one hour.\n\n"
        "For your safety, our staff will never ask for your password by phone or email."
    ),
    "Shipping & Delivery": (
        "Coffee is roasted to order and dispatched within 2 business days. Standard "
        "delivery arrives in 3 to 5 days; express arrives the next day.\n\n"
        "We currently ship only within the United States. International delivery is "
        "not yet available."
    ),
    "Store Hours": (
        "Our support team is available Monday to Friday, 9am to 6pm Eastern Time. "
        "The Austin roastery welcomes visitors on Saturdays from 10am to 2pm."
    ),
    "Subscriptions": (
        "A subscription delivers fresh coffee on a schedule you choose: weekly, "
        "biweekly, or monthly. You can pause or cancel anytime from your dashboard "
        "with no fee."
    ),
}

print(f"{len(documents)} documents loaded.")

## 3. Chunking — split documents into bite-size pieces

Why? You retrieve and send only the *relevant* pieces, not whole files — smaller chunks mean
more precise retrieval. Here we split on paragraphs; long real-world docs get split into
fixed-size windows. We keep the title on each chunk so the model knows where the text came from.

In [ ]:
chunks = []
for title, text in documents.items():
    for para in text.strip().split("\n\n"):
        chunks.append(f"{title} — {para.strip()}")

print(f"{len(documents)} documents -> {len(chunks)} chunks\n")
for c in chunks:
    print("*", c[:75], "...")

## 4. Indexing — turn each chunk into a vector (embedding)

An **embedding model** maps text to a list of numbers (here, 384) that captures its *meaning*.
Similar meaning → nearby vectors. We do this **once**, upfront.

We pass `normalize_embeddings=True` so that later we can measure similarity with a simple dot product.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # downloads on first run

chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)
print("Each chunk is now a vector of shape:", chunk_embeddings.shape)   # (n_chunks, 384)

## 5. Retrieval — semantic search

To answer a question we embed it too, then find the chunks whose vectors are **closest**.
Because the vectors are normalized, cosine similarity is just a **dot product**. We keep the top-k.

Watch the test below: the query says *"money back"*, the document says *"refund"* / *"return"* —
**no shared words**, yet it matches by meaning. That's what keyword search can't do.

In [ ]:
import numpy as np

def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_embeddings @ q          # cosine similarity (vectors are normalized)
    top = scores.argsort()[::-1][:k]       # indices of the k highest scores
    return [(chunks[i], float(scores[i])) for i in top]

for chunk, score in retrieve("how do I get my money back?", k=2):
    print(f"{score:.3f}  {chunk}")

## 6. Wrap retrieval as a tool

Same as `search_web`: a plain function that returns text Claude can read.

In [ ]:
def search_docs(query, k=3):
    hits = retrieve(query, k)
    if not hits:
        return "No relevant documents found."
    return "\n\n".join(f"[relevance {s:.2f}] {c}" for c, s in hits)

# quick sanity check
print(search_docs("when are you open?", k=1))

## 7. Schema + registry

The `query` argument is declared in the schema — the one difference from a no-argument tool.

In [ ]:
tools = [
    {
        "name": "search_docs",
        "description": (
            "Search the Brewhaus company documents for relevant information. Use this "
            "for ANY question about Brewhaus policies, orders, shipping, accounts, "
            "hours, or subscriptions."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to look up, e.g. 'refund policy' or 'shipping times'.",
                },
            },
            "required": ["query"],
        },
    },
]

TOOL_FUNCS = {"search_docs": search_docs}

## 8. The agent loop

All your notebook-01 lessons in one place: a **system prompt** that tells it *when* to search,
and **try/except** so a tool failure becomes feedback instead of a crash (`is_error: True`
marks a failed result for the model).

In [ ]:
SYSTEM_PROMPT = (
    "You are a friendly support assistant for Brewhaus, an online coffee company.\n"
    "\n"
    "Rules:\n"
    "- For ANY question about Brewhaus (refunds, orders, shipping, accounts, hours, "
    "subscriptions), call `search_docs` to find the answer. Do NOT answer from memory.\n"
    "- Base your answer only on what `search_docs` returns. If the documents do not "
    "contain the answer, say you do not have that information rather than guessing.\n"
    "- Keep answers short and warm.\n"
)

def ask(question, verbose=True):
    messages = [{"role": "user", "content": question}]
    while True:
        response = client.messages.create(
            model=model, system=SYSTEM_PROMPT, messages=messages,
            tools=tools, max_tokens=1024,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            break

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"   [tool] {block.name}({block.input})")
                try:
                    result = TOOL_FUNCS[block.name](**block.input)
                    content, is_error = str(result), False
                except Exception as e:
                    content, is_error = f"Error: {type(e).__name__}: {e}", True
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": content,
                    "is_error": is_error,
                })
        messages.append({"role": "user", "content": tool_results})

    return "".join(b.text for b in response.content if b.type == "text")

## 9. Try it — watch it retrieve, then answer

The `[tool]` line shows the agent choosing `search_docs` on its own, then grounding its
answer in what came back. The last two questions test the interesting edges:
*"Canada"* (docs say US-only) and *"tea"* (not in the docs at all → it should decline, not invent).

In [ ]:
for q in [
    "How do I get my money back?",
    "I forgot my login details, can you help?",
    "Do you deliver to Canada?",
    "Do you sell tea?",
]:
    print(f"\nUSER: {q}")
    print("AGENT:", ask(q))

## What you built + where to go next

You now have **agentic RAG**: the agent decides when to call `search_docs`, retrieves by
*meaning*, and grounds its answer in your documents.

The pipeline, start to finish:
1. **Chunk** your documents
2. **Embed** each chunk once (indexing)
3. Per question: **embed the query → retrieve top-k** nearest chunks
4. Hand them to Claude as a **tool result** → it **generates** a grounded answer

**Use your own files** instead of the toy dict — make a `knowledge_base/` folder of `.txt`
files, then:
```python
from pathlib import Path
documents = {p.stem: p.read_text(encoding="utf-8")
             for p in Path("knowledge_base").glob("*.txt")}
```
(then re-run the chunk + embed cells)

**Exercises**
- Ask something the docs don't cover and confirm it says *"I don't know"* instead of inventing
  an answer — that's **grounding**, the whole point of RAG.
- Change `k` in `retrieve` — how does more/less context change the answers?
- Print the retrieved chunks for a question so you can *see* exactly what the model was given.
- When your chunk list gets large, graduate the vector store to `chromadb` (a real vector database).
- Swap the local embedder for a hosted one (e.g. Voyage AI) for higher-quality retrieval.